# 04 — Entrenamiento con augmentation (Fase 3)

**Objetivo:** mejorar el baseline de Fase 2 probando técnicas de augmentation.

Plan de Fase 3 (orden de pruebas):
1. **Mixup sobre embeddings** (esta corrida) — barato, no requiere re-extraer embeddings.
2. *Audio augmentation* (audiomentations sobre waveform → re-extraer embeddings → `embeddings:v1`) — pendiente.
3. *Combinación* (audio aug + mixup) — pendiente.

**Target:** macro-F1 ≥ 0.80 en `test_hard`.

**Logging:** todas las corridas van a W&B con tag `phase3`. Comparar contra `baseline-v0` en la UI de W&B.

## 1. Setup del entorno Colab

Clonamos el repo desde GitHub e instalamos dependencias.

> Si ya clonaste antes y querés actualizar, corré la celda igual — hace `git pull` si la carpeta ya existe.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Placaflaca00/bird_classifierPY.git"
REPO_DIR = Path("/content/birdClassifier")

if REPO_DIR.exists():
    print("Repo ya existe, haciendo git pull...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    print("Clonando repo...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f"\ncwd: {os.getcwd()}")

In [ ]:
# Instalar el package en modo editable + deps de training.
# El pyproject.toml tiene dependencies=[] a propósito (cada surface maneja
# las suyas: lambda/, app/, training notebooks). Acá listamos las de training.
# Tarda ~2-3 min la primera vez.
!pip install -q -e ".[data]" lightning torchmetrics wandb

## 2. Credenciales W&B

Leemos `WANDB_API_KEY` y `WANDB_ENTITY` desde **Colab Secrets** (ícono 🔑 en la sidebar izquierda → "Add new secret").

Secretos que tenés que crear una sola vez:
- `WANDB_API_KEY` → tu key de https://wandb.ai/authorize
- `WANDB_ENTITY`  → tu username/team de W&B

El `WANDB_PROJECT` es público (`bird-classifierPy`), va hardcoded.

> **Nunca pegues la API key en una celda.** Si commiteás el notebook con la key adentro, queda en el historial de git para siempre.

In [ ]:
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_ENTITY"] = userdata.get("WANDB_ENTITY")
os.environ["WANDB_PROJECT"] = "bird-classifierPy"

print("WANDB_PROJECT:", os.environ["WANDB_PROJECT"])
print("WANDB_ENTITY:", os.environ["WANDB_ENTITY"])
print("WANDB_API_KEY:", "***" + os.environ["WANDB_API_KEY"][-4:])  # solo últimos 4 chars

## 3. Descargar artifacts (embeddings + splits)

`data/processed/` está gitignored: los datos no vienen con `git clone`. Vienen de **W&B Artifacts** (la fuente de verdad versionada).

Bajamos:
- `embeddings:v0` → `data/processed/embeddings.parquet` (los 1024-dim de BirdNET)
- `splits:v0`     → `data/processed/splits.parquet` (folds train/val/test_clean/test_hard)

`wandb.Api()` no abre una run nueva — solo descarga. La run real la abrirá `train()` después, y dentro registra estos mismos artifacts como input vía `use_artifact()` para mantener el lineage.

In [ ]:
import wandb

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

entity = os.environ["WANDB_ENTITY"]
project = os.environ["WANDB_PROJECT"]

api = wandb.Api()
for ref, expected_file in [
    ("embeddings:v0", "embeddings.parquet"),
    ("splits:v0",     "splits.parquet"),
]:
    full_ref = f"{entity}/{project}/{ref}"
    print(f"Bajando {full_ref}...")
    art = api.artifact(full_ref)
    art.download(root=str(PROCESSED_DIR))
    target = PROCESSED_DIR / expected_file
    if not target.exists():
        raise FileNotFoundError(f"Esperaba {target} tras descargar {ref}")
    size_mb = target.stat().st_size / 1024 / 1024
    print(f"  -> {target} ({size_mb:.1f} MB)")

print("\nContenido de data/processed/:")
for p in sorted(PROCESSED_DIR.iterdir()):
    print(f"  {p.name}")

## 4. Run: mixup_alpha = 0.2

Primera prueba: mixup conservador (`α=0.2`, recomendado por el paper original de Zhang et al. 2018 para datasets chicos).

Reusa **`embeddings:v0`** (los mismos del baseline) — el cambio es 100% en el training loop, no se re-extrae nada.

Hyperparams: idénticos al baseline excepto `mixup_alpha`. Así el delta en macro-F1 es atribuible a mixup y no a otra cosa.

In [ ]:
from src.training.train import train

result_mixup_02 = train({
    "mixup_alpha": 0.2,
    "run_name": "baseline-v0-mixup-0.2",
    "tags": ["phase3", "mixup", "alpha-0.2"],
})

print("\n=== Resumen ===")
print(f"Best val_macro_f1: {result_mixup_02['best_val_macro_f1']:.4f}")
for fold, metrics in result_mixup_02["final_results"].items():
    print(f"  {fold:<11} macro_f1={metrics['test_macro_f1']:.4f}  acc={metrics['test_acc']:.4f}")

## 5. Comparación con baseline

Ir a https://wandb.ai/<tu-entity>/bird-classifierPy y comparar las dos runs lado a lado:
- `baseline-v0` (sin mixup)
- `baseline-v0-mixup-0.2` (esta corrida)

**Métricas a mirar:**
- `final/test_hard_macro_f1` — la métrica primaria (target ≥ 0.80).
- `final/val_macro_f1` — sanity check (no debería bajar).
- Curvas de `train_loss` vs `val_loss` — con mixup el train loss típicamente sube (porque el modelo ve ejemplos "más difíciles") pero val loss debería bajar o quedar igual. Si val loss sube → mixup está hiriendo, no ayudando.

**Próximos pasos** (pendientes según resultado):
- Si mejora: probar `mixup_alpha ∈ {0.4, 1.0}` para ver si conviene más fuerte.
- Si no mejora: pasar directo a Pieza 2 (audio augmentation con audiomentations).

## 6. Run: audio augmentation (waveform K=2)

Esta corrida usa **`embeddings:aug-k2`** — embeddings generados a partir de waveforms aumentados con `audiomentations` (Gaussian SNR 15-30 dB + Shift +/-30% + Gain +/-6 dB), K=2 copias por audio del fold `train`. `test_hard` queda intacto.

Hyperparams: identicos al baseline. **Mixup desactivado** (`mixup_alpha=0`) - la corrida anterior con mixup empeoro el modelo. La intencion de este run es medir el efecto puro de waveform-aug contra el baseline.

> **Importante:** sobrescribimos `data/processed/embeddings.parquet` con el artifact aumentado. Si despues volves a correr la celda 4 (mixup_alpha=0.2), tenes que descargar de nuevo `embeddings:v0` primero.


In [ ]:
import shutil
from pathlib import Path

from src.training.train import train

# Borrar el parquet :v0 que dejo la celda 7 para que el download del aug no falle
target = PROCESSED_DIR / "embeddings.parquet"
target.unlink(missing_ok=True)

aug_ref = f"{entity}/{project}/embeddings:aug-k2"
print(f"Bajando {aug_ref}...")
art = api.artifact(aug_ref)
art.download(root=str(PROCESSED_DIR))
print(f"  -> {target} ({target.stat().st_size / 1024 / 1024:.1f} MB)")

import pandas as pd
df = pd.read_parquet(target)
n_aug = int(df["is_aug"].sum()) if "is_aug" in df.columns else 0
print(f"  filas: {len(df)}, aug rows: {n_aug}")

result_aug_k2 = train({
    "mixup_alpha": 0.0,
    "embeddings_artifact": "embeddings:aug-k2",
    "run_name": "aug-waveform-k2-cons",
    "tags": ["phase3", "audio-aug-k2", "no-mixup"],
})

print("
=== Resumen ===")
print(f"Best val_macro_f1: {result_aug_k2['best_val_macro_f1']:.4f}")
for fold, metrics in result_aug_k2["final_results"].items():
    print(f"  {fold:<11} macro_f1={metrics['test_macro_f1']:.4f}  acc={metrics['test_acc']:.4f}")


## 7. Run: baseline + class weighting (sin aug)

Volvemos a `embeddings:v0` (baseline, sin aug) y aplicamos **class weighting**
estilo sklearn-balanced sobre la cross-entropy del fold `train`. La idea es
atacar directo el desbalance que hace que macro-F1 quede bajo: clases con
menos audios (eudromia/rhea/pipile) reciben pesos más altos en la loss.

**Solo train_loss** se pesa. `val_loss` y `test_loss` quedan sin pesar para
que las métricas de comparación contra `baseline-v0` se mantengan en la misma
escala. macro-F1 y accuracy no usan pesos en ningún caso.

Esta celda también sobrescribe `data/processed/embeddings.parquet` — si la
celda anterior dejó `aug-k2`, acá lo reemplazamos por `v0`. Además, ahora
`train()` loggea automáticamente **matrices de confusión** para `test_clean`
y `test_hard` en W&B (panel `confmat/*`).

In [ ]:
from src.training.train import train

# La celda 11 sobrescribió embeddings.parquet con aug-k2. Volvemos a v0.
target = PROCESSED_DIR / "embeddings.parquet"
target.unlink(missing_ok=True)

v0_ref = f"{entity}/{project}/embeddings:v0"
print(f"Bajando {v0_ref}...")
art = api.artifact(v0_ref)
art.download(root=str(PROCESSED_DIR))
print(f"  -> {target} ({target.stat().st_size / 1024 / 1024:.1f} MB)")

result_classw = train({
    "mixup_alpha": 0.0,
    "use_class_weights": True,
    "embeddings_artifact": "embeddings:v0",
    "run_name": "baseline-v0-classw",
    "tags": ["phase3", "class-weighting", "no-aug", "no-mixup"],
})

print("
=== Resumen ===")
print(f"Best val_macro_f1: {result_classw['best_val_macro_f1']:.4f}")
for fold, metrics in result_classw["final_results"].items():
    print(f"  {fold:<11} macro_f1={metrics['test_macro_f1']:.4f}  acc={metrics['test_acc']:.4f}")

## 8. Run: baseline + class weighting + drop Eudromia formosa

Eliminamos `Eudromia formosa` del mapping y los datasets (los artifacts v0
quedan intactos; el filtrado vive en config). Razones:
- En `test_hard` tiene **0 muestras** (no se evalúa).
- En `test_clean` tiene **0 muestras** (tampoco).
- En todo el dataset hay solo 11 audios. El modelo aprendía cero útil.
- WikiAves NO la tiene (no es brasileña). Agotamos las fuentes disponibles.

Mantenemos class weighting (B). Vamos con 22 clases en lugar de 23.

In [ ]:
from src.training.train import train

# La celda 14 dejó embeddings:v0 en disco. Si volviste a correr una celda con aug,
# re-bajalo antes (mismo procedimiento que cell 14).
result_no_eudromia = train({
    "mixup_alpha": 0.0,
    "use_class_weights": True,
    "drop_species": ("Eudromia formosa",),
    "embeddings_artifact": "embeddings:v0",
    "run_name": "classw-no-eudromia",
    "tags": ["phase3", "class-weighting", "drop-eudromia", "22-classes"],
})

print("
=== Resumen ===")
print(f"Best val_macro_f1: {result_no_eudromia['best_val_macro_f1']:.4f}")
for fold, metrics in result_no_eudromia["final_results"].items():
    print(f"  {fold:<11} macro_f1={metrics['test_macro_f1']:.4f}  acc={metrics['test_acc']:.4f}")